# Credit Card Fraud Detection

Comparing three machine learning models — Logistic Regression, Random Forest, and Gradient Boosting — on the Kaggle Credit Card Fraud dataset.

**Dataset:** [Kaggle Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
Download `creditcard.csv` and place it in the same folder as this notebook before running.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score,
    average_precision_score
)

print('Libraries imported successfully.')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('creditcard.csv')

print(f'Rows       : {len(df):,}')
print(f'Columns    : {df.shape[1]}')
print(f'Fraud cases: {df["Class"].sum():,} ({df["Class"].mean()*100:.3f}%)')
print(f'Missing    : {df.isnull().sum().sum()}')
df.head(3)

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Class distribution
counts = df['Class'].value_counts()
axes[0].bar(['Legitimate', 'Fraud'], counts.values, color=['#2196F3', '#F44336'], edgecolor='black')
axes[0].set_title('Class Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2000, f'{v:,}', ha='center', fontweight='bold', fontsize=9)

# Average transaction amount by class
df.groupby('Class')['Amount'].mean().plot(
    kind='bar', ax=axes[1], color=['#2196F3', '#F44336'], edgecolor='black', rot=0
)
axes[1].set_title('Avg Transaction Amount', fontweight='bold')
axes[1].set_xticklabels(['Legitimate', 'Fraud'])
axes[1].set_ylabel('Amount (EUR)')

# Fraud amount distribution
fraud_amounts = df[df['Class'] == 1]['Amount']
axes[2].hist(fraud_amounts, bins=30, color='#F44336', edgecolor='black', alpha=0.8)
axes[2].set_title('Fraud Transaction Amounts', fontweight='bold')
axes[2].set_xlabel('Amount (EUR)')
axes[2].set_ylabel('Count')

plt.suptitle('Exploratory Data Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. Preprocessing

In [ ]:
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled']   = scaler.fit_transform(df[['Time']])
df_model = df.drop(columns=['Amount', 'Time'])

X = df_model.drop(columns=['Class'])
y = df_model['Class']

# Sample 50% of data for faster training — still 140,000+ rows
X_sample, _, y_sample, _ = train_test_split(X, y, train_size=0.5, random_state=42, stratify=y)

X_train, X_test, y_train, y_test = train_test_split(
    X_sample, y_sample, test_size=0.2, random_state=42, stratify=y_sample
)

print(f'Training rows : {len(X_train):,}')
print(f'Test rows     : {len(X_test):,}')
print(f'Fraud in train: {y_train.sum()} ({y_train.mean()*100:.3f}%)')

## 4. Model Training

In [ ]:
print('Training Logistic Regression...')
lr = LogisticRegression(class_weight='balanced', max_iter=500, random_state=42)
lr.fit(X_train, y_train)
print('  Done.')

print('Training Random Forest...')
rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
print('  Done.')

print('Training Gradient Boosting...')
gbm = HistGradientBoostingClassifier(
    max_iter=50,
    max_depth=4,
    learning_rate=0.1,
    class_weight='balanced',
    random_state=42
)
gbm.fit(X_train, y_train)
print('  Done.')

print('All 3 models trained.')

## 5. Model Evaluation

In [ ]:
models = {'Logistic Regression': lr, 'Random Forest': rf, 'Gradient Boosting': gbm}
results = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'y_pred': y_pred,
        'y_prob': y_prob,
        'f1'    : f1_score(y_test, y_pred),
        'auc'   : roc_auc_score(y_test, y_prob),
        'ap'    : average_precision_score(y_test, y_prob)
    }

summary = pd.DataFrame({
    name: {
        'F1 Score'     : round(r['f1'], 4),
        'ROC-AUC'      : round(r['auc'], 4),
        'Avg Precision': round(r['ap'], 4)
    }
    for name, r in results.items()
}).T

print('Model Comparison')
print(summary)
print(f'\nBest model by F1: {summary["F1 Score"].idxmax()}')

## 6. Visualizations

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cmaps = ['Blues', 'Reds', 'Greens']
for ax, (name, res), cmap in zip(axes, results.items(), cmaps):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
    ax.set_title(f'{name}\nF1={res["f1"]:.3f} | AUC={res["auc"]:.3f}', fontweight='bold')
plt.suptitle('Confusion Matrices', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ROC curves
plt.figure(figsize=(8, 6))
colors = ['#2196F3', '#F44336', '#4CAF50']
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={res["auc"]:.3f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
plt.title('ROC Curves', fontweight='bold')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Classification Report (Best Model)

In [ ]:
best_name = max(results, key=lambda k: results[k]['f1'])
print(f'{best_name} — Classification Report')
print(classification_report(y_test, results[best_name]['y_pred'],
                             target_names=['Legitimate', 'Fraud']))

## 8. Feature Importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
top15 = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 5))
colors = ['#F44336'] * 5 + ['#2196F3'] * 10
top15.plot(kind='bar', color=colors, edgecolor='black', linewidth=0.5)
plt.title('Top 15 Feature Importances (Random Forest)', fontweight='bold')
plt.ylabel('Importance Score')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Top 5 features:')
print(top15.head().to_string())

## 9. Save Model

In [ ]:
best_name  = max(results, key=lambda k: results[k]['f1'])
best_model = {'Logistic Regression': lr, 'Random Forest': rf, 'Gradient Boosting': gbm}[best_name]

joblib.dump(best_model, 'fraud_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print(f'Best model saved: {best_name}')
print('Files saved: fraud_model.pkl, scaler.pkl')

# Verify saved model
loaded = joblib.load('fraud_model.pkl')
sample = X_test.iloc[[0]]
pred = loaded.predict(sample)[0]
prob = loaded.predict_proba(sample)[0][1]
print(f'\nVerification prediction: {"FRAUD" if pred == 1 else "LEGITIMATE"} (confidence: {prob:.4f})')

## 10. Predict Single Transaction

In [ ]:
def predict_transaction(transaction_dict):
    """Predict fraud probability for a single transaction."""
    model = joblib.load('fraud_model.pkl')
    df_in = pd.DataFrame([transaction_dict])
    pred  = model.predict(df_in)[0]
    prob  = model.predict_proba(df_in)[0][1]
    return {
        'is_fraud'         : bool(pred),
        'fraud_probability': round(float(prob), 4),
        'decision'         : 'BLOCK' if pred == 1 else 'APPROVE'
    }

legit_row = X_test[y_test == 0].iloc[0].to_dict()
fraud_row = X_test[y_test == 1].iloc[0].to_dict()

print('Legitimate transaction:')
print(predict_transaction(legit_row))

print('\nFraud transaction:')
print(predict_transaction(fraud_row))

## 11. FastAPI Deployment

In [ ]:
api_code = '''from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import pandas as pd

app = FastAPI(title="Credit Card Fraud Detection API")
model = joblib.load("fraud_model.pkl")

class Transaction(BaseModel):
    V1: float; V2: float; V3: float; V4: float; V5: float
    V6: float; V7: float; V8: float; V9: float; V10: float
    V11: float; V12: float; V13: float; V14: float; V15: float
    V16: float; V17: float; V18: float; V19: float; V20: float
    V21: float; V22: float; V23: float; V24: float; V25: float
    V26: float; V27: float; V28: float
    Amount_scaled: float; Time_scaled: float

@app.get("/")
def root():
    return {"message": "Fraud Detection API is running."}

@app.post("/predict")
def predict(t: Transaction):
    data = pd.DataFrame([t.dict()])
    pred = model.predict(data)[0]
    prob = model.predict_proba(data)[0][1]
    return {
        "is_fraud": bool(pred),
        "fraud_probability": round(float(prob), 4),
        "decision": "BLOCK" if pred == 1 else "APPROVE"
    }
'''

with open('app.py', 'w') as f:
    f.write(api_code)

print('app.py saved.')
print()
print('To run the API:')
print('  pip install fastapi uvicorn')
print('  uvicorn app:app --reload')
print()
print('Then open: http://localhost:8000/docs')

## Summary

| Step | Description |
|------|-------------|
| EDA | Class imbalance visualization, amount distribution |
| Preprocessing | Scaled Amount and Time, stratified 80/20 split |
| Models | Logistic Regression, Random Forest, Gradient Boosting |
| Evaluation | F1 Score, ROC-AUC, Confusion Matrix, Classification Report |
| Feature Importance | Top 15 features from Random Forest |
| Deployment | Model saved as `.pkl`, FastAPI endpoint in `app.py` |